# 02 Explore the sensor data

Day 1, 11:15 to 12:15. **Goal:** understand the data well enough to write **3 engineering observations**
an operations manager would find useful.

The business question: *which machine runs are about to fail?* Unplanned downtime is expensive,
so we want to see failures coming.

In [ ]:
from pathlib import Path
import pandas as pd

# Works whether Jupyter starts in the repo root or in notebooks/
DATA = next(p for p in [Path("data/raw/ai4i2020.csv"), Path("../data/raw/ai4i2020.csv")] if p.exists())
df = pd.read_csv(DATA)
import matplotlib.pyplot as plt

print(df.shape)
df.head()

## 1. First look: types, ranges, missing values

`info()` shows types and missing values; `describe()` shows ranges. Check the ranges make physical sense
(temperatures are in **kelvin**, around 300 K = 27 °C).

In [ ]:
df.info()

In [ ]:
df.describe().round(1).T

In [ ]:
df.isna().sum().sum()   # total missing values in the whole table

## 2. How often do machines fail?

This one number shapes everything we do today.

In [ ]:
# TODO: count failures vs non-failures with value_counts(), and print the failure rate as a %
counts = ...
rate = ...

> **Pause and predict.** If a "model" just said *no failure* for every run, what accuracy would it get?
> Write your guess, then check: `1 - rate`.

In [ ]:
print(f"'Always no failure' accuracy: {1 - rate:.2%}")

## 3. Failure modes: *how* do machines fail?

Five columns record the mode of each failure:

| Column | Mode |
|---|---|
| `twf` | tool wear failure |
| `hdf` | heat dissipation failure |
| `pwf` | power failure |
| `osf` | overstrain failure |
| `rnf` | random failure |

In [ ]:
modes = ["twf", "hdf", "pwf", "osf", "rnf"]
# TODO: sum each mode column, sort, and draw a bar chart
mode_counts = ...

Some runs fail in more than one way at once:

In [ ]:
n_modes = df[modes].sum(axis=1)
print("failures with 2+ modes:", (n_modes > 1).sum())

### Label quirks (real data is never perfectly clean)

Check whether the target and the mode columns always agree.

In [ ]:
fail_no_mode = ((df["machine_failure"] == 1) & (n_modes == 0)).sum()
mode_no_fail = ((df["machine_failure"] == 0) & (n_modes > 0)).sum()
print("failure recorded but no mode:", fail_no_mode)
print("mode recorded but no failure:", mode_no_fail)
df.loc[(df["machine_failure"] == 0) & (n_modes > 0), modes].sum()

**What this means:** 9 failures have no recorded cause, and 18 runs are flagged as *random failure*
without a machine failure. The dataset author designed random failures this way. We leave the labels
as they are, but we would raise this with the data owner. On Day 4 we turn checks like this into
automatic **data validation**.

### Why we will never use the mode columns as model inputs

The mode is written down **after** a failure happens. At prediction time it does not exist yet.
Feeding it to a model is called **target leakage**: the model would look brilliant in testing
and be useless on the shop floor. We drop `twf, hdf, pwf, osf, rnf` (and the ID columns) before modelling.

## 4. Does product quality matter?

`type` is the product quality variant: L (low), M (medium), H (high).

In [ ]:
# TODO: group by type and get the number of runs and the failure rate for each
by_type = ...

## 5. Torque and tool wear vs failure

Box plots compare the spread of a reading for runs that failed vs runs that did not.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.boxplot(column="torque_nm", by="machine_failure", ax=axes[0])
df.boxplot(column="tool_wear_min", by="machine_failure", ax=axes[1])
axes[0].set_title("Torque (Nm)")
axes[1].set_title("Tool wear (min)")
for ax in axes:
    ax.set_xlabel("machine_failure (0 = ok, 1 = failed)")
plt.suptitle("")
plt.tight_layout()
plt.show()

Failure rate by tool-wear band makes the pattern easier to read than a box plot:

In [ ]:
bands = pd.cut(df["tool_wear_min"], bins=[0, 50, 100, 150, 200, 260], include_lowest=True)
# TODO: failure rate (%) for each band

## 6. Torque and speed together

Each dot is a run; red dots failed. Look at the *edges* of the cloud.

In [ ]:
ok = df[df["machine_failure"] == 0]
bad = df[df["machine_failure"] == 1]
plt.figure(figsize=(7, 5))
plt.scatter(ok["rotational_speed_rpm"], ok["torque_nm"], s=3, alpha=0.2, label="ok", color="grey")
plt.scatter(bad["rotational_speed_rpm"], bad["torque_nm"], s=10, alpha=0.8, label="failed", color="crimson")
plt.xlabel("rotational speed (rpm)")
plt.ylabel("torque (Nm)")
plt.legend()
plt.title("Failures sit at the extremes of torque and speed")
plt.show()

## 7. Two engineered features

Engineers think in **power** and **temperature difference**, not raw readings. Let's compute both.

- `power_w = torque_nm × rotational_speed_rpm × 2π / 60`
- `temp_diff_k = process_temp_k − air_temp_k` (small difference = poor heat dissipation)

In [ ]:
import numpy as np

# TODO: create the two new columns, then compare them for failed vs ok runs
df["power_w"] = ...
df["temp_diff_k"] = ...

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["power_w", "temp_diff_k"]):
    ax.hist(df.loc[df.machine_failure == 0, col], bins=50, alpha=0.5, density=True, label="ok")
    ax.hist(df.loc[df.machine_failure == 1, col], bins=50, alpha=0.6, density=True, label="failed", color="crimson")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Your 3 engineering observations

Write three observations an operations manager could act on. A good observation is
**specific, backed by a number from this notebook, and says why it matters**.

- Weak: *"Tool wear matters."*
- Strong: *"Runs with tool wear above 200 minutes fail at roughly X%, about N times the average; a tool-change rule near 200 minutes could prevent a share of failures."*

1. ...
2. ...
3. ...